# CEO Letter / Annual Report Event Study — US Insurers (First Pass)

Tests whether abnormal stock returns around CEO letter/annual report publication dates correlate
with ESG narrative specificity. **US-region firms only for this first pass** (7 companies x 12
fiscal years = 84 firm-years). Kept **fully separate** from
`PHDp2_CEOLetters_AnnualReports_TextAnalysis.ipynb` and `PHDp2_FinancialData_Regression.ipynb` --
this notebook does not import from or modify either. Not validated yet -- developed on branch
`event-study`, not `main`.

**Report-only pipeline.** No merge with specificity measures, no regression -- Step 5 ends with a
confirmed firm-year event-study panel, full stop.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# pip installs yfinance if not already present in this Colab runtime
try:
    import yfinance as yf
except ImportError:
    %pip install -q yfinance
    import yfinance as yf

print("yfinance version:", yf.__version__)


## Step 1 — Load event dates

Read `Combined_Insurer_Publication_Dates.xlsx`, filter to `Region == "United States"`, confirm 84
firm-years (7 companies x 12 fiscal years) before proceeding. Chubb's rows already unify the
2016 ACE Limited -> Chubb Limited name change into a single `"Chubb"` company label in the source
file (confirmed directly: FY2012-2015 rows carry a note "Filed as ACE Limited pre-2016 merger,"
FY2016-2023 rows carry "Filed as Chubb Limited" -- both already grouped under `Company == "Chubb"`
with `Region == "United States"`), so no company-name unification step is needed here -- it's
already done in the source data. The ticker-level ACE/CB split is handled in Step 2/3 instead,
since that's a market-data concern, not an event-date concern.

In [ ]:
# Adjust this path if the file lives elsewhere in your Drive.
event_dates_path = "/content/drive/MyDrive/phd/Data/Combined_Insurer_Publication_Dates.xlsx"

df_dates_raw = pd.read_excel(event_dates_path)
print(f"Full file: {df_dates_raw.shape}")
print("Regions present:", sorted(df_dates_raw['Region'].unique()))

df_us = df_dates_raw[df_dates_raw["Region"] == "United States"].copy()
print(f"\nUS-region rows: {len(df_us)}")

# ------------------------------------------------------------
# Confirm 84 firm-years (7 companies x 12 fiscal years) before proceeding,
# per instruction -- stop and investigate if this doesn't hold, rather
# than silently continue on a wrong filter.
# ------------------------------------------------------------
n_companies = df_us["Company"].nunique()
n_years = df_us["Fiscal Year"].nunique()
print(f"Distinct companies: {n_companies}")
print(f"Distinct fiscal years: {n_years}")
print(f"Companies: {sorted(df_us['Company'].unique())}")
print(f"Fiscal years: {sorted(df_us['Fiscal Year'].unique())}")

counts_per_company = df_us["Company"].value_counts()
print("\nRows per company:")
print(counts_per_company)

assert len(df_us) == 84, f"Expected 84 US firm-years, got {len(df_us)} -- STOP, investigate before continuing."
assert n_companies == 7, f"Expected 7 US companies, got {n_companies}"
assert (counts_per_company == 12).all(), "Not every company has exactly 12 fiscal years -- STOP."
print("\nCONFIRMED: 84 US firm-years (7 companies x 12 fiscal years).")

# ------------------------------------------------------------
# Parse Publication Date (stored as a string in the source file, not a
# native Excel date) and sanity-check the Chubb/ACE transition boundary
# noted above.
# ------------------------------------------------------------
df_us["Publication Date"] = pd.to_datetime(df_us["Publication Date"])
print(f"\nPublication date range: {df_us['Publication Date'].min()} to {df_us['Publication Date'].max()}")

chubb = df_us[df_us["Company"] == "Chubb"][["Fiscal Year", "Publication Date", "Note"]].sort_values("Fiscal Year")
print("\nChubb rows (confirm ACE->Chubb transition sits between FY2015 and FY2016):")
print(chubb.to_string(index=False))


## Step 2 — Tickers

Company -> ticker mapping, plus the benchmark (`URTH`, iShares MSCI World ETF). Chubb needs
special handling: pre-2016 it traded as **ACE Limited** under ticker **ACE**, not `CB`. Rather
than assume where `CB`'s history starts, this pulls both `ACE` and `CB` and empirically checks
where each series actually starts/ends -- the actual data boundary, not an assumed merger-closing
date, decides the splice point (handled in Step 3).

In [ ]:
COMPANY_TICKER = {
    "American International Group (AIG)": "AIG",
    "Chubb": "CB",                     # current ticker; ACE spliced in for pre-merger history
    "MetLife, Inc.": "MET",
    "Prudential Financial, Inc.": "PRU",
    "The Allstate Corporation": "ALL",
    "The Progressive Corporation": "PGR",
    "The Travelers Companies, Inc.": "TRV",
}
ACE_TICKER = "ACE"          # Chubb's pre-2016-merger ticker (as ACE Limited)
BENCHMARK_TICKER = "URTH"   # iShares MSCI World ETF

# ACE is checked SEPARATELY and non-fatally below -- it's Chubb's
# pre-2016 ticker, and Yahoo Finance frequently drops a ticker's price
# history entirely after a symbol change/delisting rather than
# archiving it, so ACE failing to resolve at all (confirmed: fails even
# against a 2014 window, when it was definitely actively trading) is a
# real possible outcome, not necessarily a bug to "fix." The other 8
# tickers (7 companies' current symbols + URTH) are the ones that must
# all resolve for this pipeline to proceed at all.
required_tickers = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
print("Required tickers to verify:", required_tickers)

DEFAULT_PROBE_WINDOW = ("2019-01-01", "2019-01-31")

ticker_resolves = {}
for ticker in required_tickers:
    start, end = DEFAULT_PROBE_WINDOW
    probe = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    ok = len(probe) > 0
    ticker_resolves[ticker] = ok
    print(f"{ticker}: {'OK' if ok else 'FAILED'} ({len(probe)} rows in {start} probe window)")

failed = [t for t, ok in ticker_resolves.items() if not ok]
if failed:
    raise RuntimeError(f"These REQUIRED tickers did not resolve in yfinance: {failed} -- STOP, fix before Step 3.")
print("\nAll required tickers resolve.")

# ------------------------------------------------------------
# ACE probe (2014, when it was definitely trading) -- reported, not
# raised on, since the real question this pipeline needs answered is
# "does SOME source cover the pre-2016 Chubb event dates," and CB's own
# history (checked next) is the other candidate.
# ------------------------------------------------------------
ace_probe = yf.download("ACE", start="2014-01-01", end="2014-01-31", progress=False, auto_adjust=True)
ace_probe_ok = len(ace_probe) > 0
print(f"\nACE (informational, not required): {'OK' if ace_probe_ok else 'NO DATA'} "
      f"({len(ace_probe)} rows in 2014-01 probe window)")

# ------------------------------------------------------------
# Check whether CB's own history already extends back far enough to
# cover the earliest Chubb event dates (2012-2015 fiscal years, i.e.
# publication dates as early as 2013). Report the actual first
# available date rather than assume -- this determines whether the
# pre-2016 Chubb firm-years are usable at all via yfinance.
# ------------------------------------------------------------
cb_full = yf.download("CB", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True)
ace_full = yf.download("ACE", start="2011-01-01", end="2016-06-30", progress=False, auto_adjust=True) if ace_probe_ok else None

print(f"\nCB: first available date = {cb_full.index.min() if len(cb_full) else 'NO DATA'}")
if ace_full is not None and len(ace_full):
    print(f"ACE: first available date = {ace_full.index.min()}, last available date = {ace_full.index.max()}")
else:
    print("ACE: NO DATA available from yfinance at all (confirmed via both the 2014 and "
          "2011-2016 pulls) -- Yahoo Finance has evidently dropped this ticker's history "
          "entirely rather than archiving it under the old symbol.")

earliest_chubb_event = chubb["Publication Date"].min()
print(f"\nEarliest Chubb event date requiring price history: {earliest_chubb_event}")
print(f"Estimation window for that event needs data back to roughly "
      f"{earliest_chubb_event - pd.Timedelta(days=380)} (250 trading days plus weekends/holidays buffer)")

CB_COVERS_PRE_2016 = len(cb_full) > 0 and cb_full.index.min() <= pd.Timestamp("2015-01-01")
ACE_AVAILABLE = ace_full is not None and len(ace_full) > 0

if CB_COVERS_PRE_2016:
    print("\nCB's own history extends back far enough -- no splice needed. VERIFY this isn't "
          "Yahoo silently truncating/misdating data rather than genuinely having ACE-era prices "
          "under the CB symbol, by spot-checking one known pre-2016 price point before trusting it.")
    CHUBB_STRATEGY = "cb_only"
elif ACE_AVAILABLE:
    print("\nCB does NOT cover pre-2016, but ACE does -- splicing ACE + CB (handled in Step 4).")
    CHUBB_STRATEGY = "splice"
else:
    print("\n" + "=" * 70)
    print("DATA GAP: neither CB nor ACE covers Chubb's pre-2016 fiscal years via yfinance.")
    print("=" * 70)
    print("This means Chubb FY2012-2015 (4 of the 84 firm-years) cannot be included in the "
          "event study through this data source. Two options, NOT decided here:")
    print("  (a) Drop those 4 Chubb firm-years, proceed with N=80 (84 - 4).")
    print("  (b) Source ACE's pre-2016 price history from a different provider (e.g. a paid "
          "data vendor, or a manually-compiled CSV) and splice it in manually.")
    print("Proceeding with CB-only for Chubb (FY2016-2023) and flagging FY2012-2015 as excluded "
          "in Step 4/5's output -- report back and we can revisit if (b) is wanted instead.")
    CHUBB_STRATEGY = "cb_only_with_gap"

print(f"\nCHUBB_STRATEGY = {CHUBB_STRATEGY!r}")


## Step 3 — Pull daily price data

All 7 tickers (+ `ACE` for the pre-2016 Chubb splice) + `URTH`, Jan 2011 (estimation-window
buffer before the earliest 2012 fiscal-year event) through Dec 2023. Raw pulls saved to disk
before any processing, so Step 4/5 can be re-run without re-hitting the API.

In [ ]:
PRICE_START = "2011-01-01"
# 2023-12-31 originally -- WRONG: FY2023 events are published in Q1/Q2
# 2024 (latest event date 2024-04-26), so that cutoff excluded every
# FY2023 event window and part of some FY2023 estimation windows too.
# Confirmed systematic across all 7 companies (not Chubb-specific) via
# the exclusion-accounting re-check in the report-only summary cell.
# Extended to give buffer past the latest event date's post-event window.
PRICE_END = "2024-07-31"

raw_prices = {}
tickers_to_pull = list(COMPANY_TICKER.values()) + [BENCHMARK_TICKER]
if CHUBB_STRATEGY == "splice":
    tickers_to_pull.append(ACE_TICKER)
else:
    print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r} -- not pulling ACE (unavailable or unneeded).")

for ticker in tickers_to_pull:
    if ticker in raw_prices:
        continue  # CB only pulled once even though it's used for one company
    print(f"Pulling {ticker}...")
    data = yf.download(ticker, start=PRICE_START, end=PRICE_END, progress=False, auto_adjust=True)
    raw_prices[ticker] = data
    print(f"  {len(data)} rows, {data.index.min()} to {data.index.max()}")

raw_prices_dir = "/content/drive/MyDrive/phd/Data/event_study_raw_prices"
os.makedirs(raw_prices_dir, exist_ok=True)
for ticker, data in raw_prices.items():
    out_path = os.path.join(raw_prices_dir, f"{ticker}.csv")
    data.to_csv(out_path)
    print(f"Saved {ticker} -> {out_path}")

print(f"\nAll raw pulls saved to: {raw_prices_dir}")


## Step 4 — Market-model event study, per firm-year

Estimation window `[-250, -30]` trading days relative to the (trading-day-adjusted) event date;
OLS of firm return on `URTH` return -> firm-specific alpha, beta. Event windows: primary
`[-1, +1]`; also `[-2, +2]` and `[0, +1]` as robustness alternatives, all three reported.
Abnormal return = actual - (alpha + beta x URTH return); CAR = sum over the window, both signed
and `|CAR|` reported. Non-trading-day publication dates shift to the next available trading day
(reported how many). Firm-years with fewer than 100 valid estimation-window trading days excluded
(reported which, if any).

In [ ]:
# ------------------------------------------------------------
# Build one continuous return series per company. For Chubb, splice ACE
# (up to ACE's last available date) and CB (from CB's first available
# date) -- using the ACTUAL empirical data boundary from Step 2, not an
# assumed merger-closing date, so the estimation window for the FY2015
# event (published 2016-04-08, whose [-250,-30] estimation window
# extends back into 2015, i.e. BEFORE the ACE->CB ticker switch) is
# built from genuinely continuous, correctly-sourced price data rather
# than silently using CB data that doesn't exist yet for that period.
# ------------------------------------------------------------
def get_adj_close(ticker):
    # yfinance can return either flat columns or MultiIndex columns
    # (ticker as a sub-level) even for a single-ticker download,
    # depending on version -- handled robustly rather than assumed flat,
    # since a silent DataFrame-vs-Series mismatch here would break
    # everything downstream without an obvious error at the point of failure.
    close = raw_prices[ticker]["Close"]  # auto_adjust=True -> Close is already adjusted
    if isinstance(close, pd.DataFrame):
        close = close.squeeze("columns")
    return close.dropna()

def build_return_series(company):
    ticker = COMPANY_TICKER[company]
    if company == "Chubb" and CHUBB_STRATEGY == "splice":
        ace_close = get_adj_close(ACE_TICKER)
        cb_close = get_adj_close("CB")
        splice_date = cb_close.index.min()
        ace_part = ace_close[ace_close.index < splice_date]
        print(f"Chubb: splicing ACE (through {ace_part.index.max()}) "
              f"+ CB (from {cb_close.index.min()})")
        gap_days = (cb_close.index.min() - ace_part.index.max()).days
        if gap_days > 10:
            print(f"  WARNING: {gap_days}-day gap between ACE's last price and CB's first price "
                  f"-- investigate before trusting the spliced return series across this boundary.")
        combined_close = pd.concat([ace_part, cb_close]).sort_index()
        combined_close = combined_close[~combined_close.index.duplicated(keep="last")]
    elif company == "Chubb":
        # CHUBB_STRATEGY is "cb_only" or "cb_only_with_gap" -- CB history
        # only, no ACE splice. In the "_with_gap" case this means
        # FY2012-2015 events will have no estimation-window data
        # available and get excluded naturally by Step 4's existing
        # "estimation window falls outside available price history"
        # check below -- not special-cased here, just a consequence of
        # only having CB's actual price history to work with.
        combined_close = get_adj_close("CB")
        print(f"Chubb: CB only (CHUBB_STRATEGY={CHUBB_STRATEGY!r}), "
              f"history from {combined_close.index.min()}")
    else:
        combined_close = get_adj_close(ticker)
    returns = combined_close.pct_change().dropna()
    returns.name = company
    return returns

benchmark_returns = get_adj_close(BENCHMARK_TICKER).pct_change().dropna()
benchmark_returns.name = "URTH"

company_returns = {company: build_return_series(company) for company in COMPANY_TICKER}


In [ ]:
# ------------------------------------------------------------
# Trading-day calendar taken directly from URTH's own observed trading
# days (rather than a generic calendar), so "next available trading day"
# and window offsets are defined consistently with the actual price data
# being used.
# ------------------------------------------------------------
trading_days = benchmark_returns.index.sort_values()

def next_trading_day(date, calendar):
    idx = calendar.searchsorted(date)
    if idx >= len(calendar):
        return None
    return calendar[idx]

def trading_day_offset(date, calendar, offset):
    """Trading day `offset` sessions away from `date` (date must be IN calendar)."""
    pos = calendar.get_loc(date)
    new_pos = pos + offset
    if new_pos < 0 or new_pos >= len(calendar):
        return None
    return calendar[new_pos]

EVENT_WINDOWS = {
    "primary_-1_+1": (-1, 1),
    "robustness_-2_+2": (-2, 2),
    "robustness_0_+1": (0, 1),
    "robustness_0_+5": (0, 5),
    "robustness_0_+20": (0, 20),
}
ESTIMATION_WINDOW = (-250, -30)
MIN_ESTIMATION_OBS = 100

results = []
n_shifted = 0
excluded_thin_estimation = []

for _, row in df_us.iterrows():
    company = row["Company"]
    fiscal_year = row["Fiscal Year"]
    raw_event_date = row["Publication Date"]

    firm_ret = company_returns[company]
    # returns are pct_change of price, indexed by the LATER of the two
    # price dates -- align against a calendar of dates where a return is
    # actually observable for this firm.
    firm_calendar = firm_ret.index

    # Shift to next available trading day if the raw event date isn't one.
    if raw_event_date in firm_calendar:
        event_date = raw_event_date
    else:
        event_date = next_trading_day(raw_event_date, firm_calendar)
        n_shifted += 1
        if event_date is None:
            print(f"SKIPPING {company} FY{fiscal_year}: no trading day found on/after "
                  f"{raw_event_date} in the available price history.")
            continue

    event_pos_check = firm_calendar.get_loc(event_date)

    # ---- estimation window ----
    est_start = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[0])
    est_end = trading_day_offset(event_date, firm_calendar, ESTIMATION_WINDOW[1])
    if est_start is None or est_end is None:
        print(f"SKIPPING {company} FY{fiscal_year}: estimation window falls outside available "
              f"price history.")
        continue

    est_mask = (firm_ret.index >= est_start) & (firm_ret.index <= est_end)
    est_firm = firm_ret[est_mask]
    est_bench = benchmark_returns.reindex(est_firm.index).dropna()
    est_common = est_firm.index.intersection(est_bench.index)
    est_firm = est_firm.loc[est_common]
    est_bench = benchmark_returns.loc[est_common]

    n_est = len(est_common)
    if n_est < MIN_ESTIMATION_OBS:
        excluded_thin_estimation.append((company, fiscal_year, n_est))
        continue

    # OLS: firm_return = alpha + beta * bench_return
    X = np.column_stack([np.ones(n_est), est_bench.values])
    coefs, _, _, _ = np.linalg.lstsq(X, est_firm.values, rcond=None)
    alpha, beta = coefs[0], coefs[1]

    # ---- event windows ----
    car_values = {}
    for win_name, (lo, hi) in EVENT_WINDOWS.items():
        win_start = trading_day_offset(event_date, firm_calendar, lo)
        win_end = trading_day_offset(event_date, firm_calendar, hi)
        if win_start is None or win_end is None:
            car_values[win_name] = np.nan
            continue
        win_mask = (firm_ret.index >= win_start) & (firm_ret.index <= win_end)
        win_firm = firm_ret[win_mask]
        win_bench = benchmark_returns.reindex(win_firm.index)
        abnormal = win_firm - (alpha + beta * win_bench)
        car_values[win_name] = abnormal.sum()

    results.append({
        "Company": company,
        "Fiscal Year": fiscal_year,
        "Event Date (raw)": raw_event_date,
        "Event Date (adjusted)": event_date,
        "Shifted": event_date != raw_event_date,
        "Estimation N": n_est,
        "Alpha": alpha,
        "Beta": beta,
        "CAR[-1,+1]": car_values["primary_-1_+1"],
        "CAR[-2,+2]": car_values["robustness_-2_+2"],
        "CAR[0,+1]": car_values["robustness_0_+1"],
        "CAR[0,+5]": car_values["robustness_0_+5"],
        "CAR[0,+20]": car_values["robustness_0_+20"],
        "|CAR|[-1,+1]": abs(car_values["primary_-1_+1"]),
    })

print(f"Publication dates shifted to next trading day: {n_shifted} of {len(df_us)}")
print(f"\nFirm-years excluded for <{MIN_ESTIMATION_OBS} estimation-window observations: "
      f"{len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: only {n} estimation-window observations")


## Step 5 — Firm-year event-study panel

`Company | Fiscal Year | Event Date (adjusted) | Estimation N | Alpha | Beta | CAR[-1,+1] |
CAR[-2,+2] | CAR[0,+1] | |CAR|[-1,+1]`. No merge with specificity measures, no regression -- this
panel, confirmed correct, is the end of this notebook's first pass.

In [ ]:
panel = pd.DataFrame(results)
panel = panel.sort_values(["Company", "Fiscal Year"]).reset_index(drop=True)

print(f"Firm-years in final panel: {len(panel)} of {len(df_us)} original US firm-years")
print(f"({len(df_us) - len(panel)} excluded -- see Step 4's exclusion list above for why)")

display_cols = ["Company", "Fiscal Year", "Event Date (adjusted)", "Estimation N",
                "Alpha", "Beta", "CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]",
                "CAR[0,+5]", "CAR[0,+20]", "|CAR|[-1,+1]"]
display(panel[display_cols])

output_path = "/content/drive/MyDrive/phd/Data/ceo_letter_event_study_panel_US.csv"
panel[display_cols].to_csv(output_path, index=False)
print(f"\nSaved panel to: {output_path}")


## Report-only summary

CAR summary stats, exclusions and why, confirmation of the Chubb/ACE handling, and a spot-check
for unadjusted stock splits (an obviously-wrong single-day return, e.g. close to -50% or +100%,
is the classic signature of a split that `auto_adjust=True` failed to correctly adjust for --
checked explicitly below rather than assumed fine).

In [ ]:
print("=" * 70)
print("CAR summary statistics")
print("=" * 70)
print(panel[["CAR[-1,+1]", "CAR[-2,+2]", "CAR[0,+1]", "CAR[0,+5]", "CAR[0,+20]",
             "|CAR|[-1,+1]"]].describe())

print(f"\n{'=' * 70}\nExclusions\n{'=' * 70}")
print(f"Original US firm-years: {len(df_us)}")
print(f"Excluded for <{MIN_ESTIMATION_OBS} estimation obs: {len(excluded_thin_estimation)}")
for company, fy, n in excluded_thin_estimation:
    print(f"  {company} FY{fy}: {n} obs")
print(f"Final panel: {len(panel)}")

# ------------------------------------------------------------
# GENERAL exclusion-accounting check, all 7 companies, not just Chubb --
# this is what actually caught the FY2023/PRICE_END bug (every company
# was silently missing exactly FY2023, not something the Chubb-specific
# check below would have caught on its own). Re-run every time this
# notebook runs, not just after a known bug, since a systematic gap like
# this produces no error/warning on its own -- rows just don't show up.
# ------------------------------------------------------------
print(f"\n{'=' * 70}\nFull exclusion accounting -- every company, every fiscal year\n{'=' * 70}")
expected_pairs = set(zip(df_us["Company"], df_us["Fiscal Year"]))
actual_pairs = set(zip(panel["Company"], panel["Fiscal Year"]))
missing_pairs = sorted(expected_pairs - actual_pairs)
unexpected_pairs = sorted(actual_pairs - expected_pairs)

print(f"Expected firm-years: {len(expected_pairs)}")
print(f"Present in final panel: {len(actual_pairs)}")

if missing_pairs:
    print(f"\nMISSING firm-years ({len(missing_pairs)}):")
    missing_by_company = {}
    for company, fy in missing_pairs:
        missing_by_company.setdefault(company, []).append(fy)
    for company, fys in missing_by_company.items():
        print(f"  {company}: {sorted(fys)}")
    # Flag whether the gap is isolated to one company (e.g. the known
    # Chubb/ACE gap) or systematic across companies (e.g. the PRICE_END
    # bug this replaced) -- these need different explanations.
    companies_affected = len(missing_by_company)
    if companies_affected > 1:
        common_fys = set.intersection(*[set(fys) for fys in missing_by_company.values()])
        if common_fys:
            print(f"\n  SYSTEMATIC: {companies_affected} companies all missing fiscal year(s) "
                  f"{sorted(common_fys)} -- check PRICE_END / event-date coverage, not a "
                  f"single-company data issue.")
else:
    print("\nNo missing firm-years.")

if unexpected_pairs:
    print(f"\nUNEXPECTED firm-years in panel not in source data ({len(unexpected_pairs)}): "
          f"{unexpected_pairs} -- investigate, this should not happen.")

assert len(panel) == 84 or (len(panel) == 80 and CHUBB_STRATEGY == "cb_only_with_gap"), (
    f"Final panel has {len(panel)} rows -- expected 84 (full) or 80 (with the known, reported "
    f"Chubb FY2012-2015 gap). Anything else means an unexplained loss -- STOP and investigate "
    f"before trusting this panel."
)
if len(panel) == 84:
    print("\nCONFIRMED: 84/84 firm-years present, no gaps.")
else:
    print(f"\nCONFIRMED: {len(panel)}/84 firm-years present, with the expected and reported "
          f"Chubb FY2012-2015 gap (CHUBB_STRATEGY={CHUBB_STRATEGY!r}) -- no OTHER unexplained gap.")

print(f"\n{'=' * 70}\nChubb / ACE ticker transition\n{'=' * 70}")
print(f"CHUBB_STRATEGY = {CHUBB_STRATEGY!r}")
chubb_panel = panel[panel["Company"] == "Chubb"]
chubb_fiscal_years_in_panel = sorted(chubb_panel["Fiscal Year"].tolist())
chubb_fiscal_years_expected = sorted(df_us[df_us["Company"] == "Chubb"]["Fiscal Year"].tolist())
chubb_missing = sorted(set(chubb_fiscal_years_expected) - set(chubb_fiscal_years_in_panel))
print(f"Chubb fiscal years in final panel: {chubb_fiscal_years_in_panel}")
if chubb_missing:
    print(f"Chubb fiscal years MISSING from final panel: {chubb_missing} "
          f"({'expected -- yfinance has no ACE-era price data, see Step 2' if CHUBB_STRATEGY == 'cb_only_with_gap' else 'unexpected, investigate'})")
else:
    print("All 12 Chubb fiscal years present.")
print(chubb_panel[["Fiscal Year", "Event Date (adjusted)"]].to_string(index=False))
if CHUBB_STRATEGY == "splice":
    print("\nSee Step 4's splice-point print for the exact ACE-end / CB-start dates used, and "
          "whether the gap-check warning fired.")
elif CHUBB_STRATEGY == "cb_only_with_gap":
    print("\nNeither CB nor ACE covers Chubb's pre-2016 history via yfinance (confirmed in "
          "Step 2) -- Chubb FY2012-2015 are excluded from this panel, not spliced or imputed. "
          "If those 4 firm-years are needed, pre-2016 ACE Limited prices would need to be "
          "sourced from a different data provider.")

print(f"\n{'=' * 70}\nSpot-check: unadjusted stock splits\n{'=' * 70}")
print("Flags any single-day return whose magnitude exceeds 20% for any of the 7 firms -- a very "
      "large one-day move that isn't obviously tied to a known market-wide shock is the classic "
      "signature of an unadjusted split slipping through (auto_adjust=True should prevent this, "
      "but checked explicitly rather than assumed).")
for company, ret in company_returns.items():
    extreme = ret[ret.abs() > 0.20]
    if len(extreme) > 0:
        print(f"\n{company}: {len(extreme)} day(s) with |return| > 20%:")
        print(extreme.to_string())
    else:
        print(f"{company}: none")
